# Algoritmos de optimización - Seminario

Nombre y Apellidos:   
* Andrés Gómez
* Iván Cañón


Url: https://github.com/Icanongonz/SEMINARIO/blob/colab/Seminario_Algoritmos.ipynb<br>
Problema: **Sesiones de doblaje**

Descripción del problema: Se precisa coordinar el doblaje de una película. Los actores del doblaje deben coincidir en las
tomas en las que sus personajes aparecen juntos en las diferentes tomas. Los actores de
doblaje cobran todos la misma cantidad por cada día que deben desplazarse hasta el estudio de
grabación independientemente del número de tomas que se graben. No es posible grabar más
de 6 tomas por día. El objetivo es planificar las sesiones por día de manera que el gasto por los
servicios de los actores de doblaje sea el menor posible                                        

(*)¿Cuantas posibilidades hay sin tener en cuenta las restricciones?<br>



¿Cuantas posibilidades hay teniendo en cuenta todas las restricciones.




Modelo para el espacio de soluciones<br>
(*) ¿Cual es la estructura de datos que mejor se adapta al problema? Argumentalo.(Es posible que hayas elegido una al principio y veas la necesidad de cambiar, arguentalo)


Respuesta: 

* La estructura elegida para los datos del problema es una matriz de *t* tomas x *a* actores, en cada elemento (toma, actor) un *1* o *0* según si el actor tiene participación o no. Es una estructura simple que nos permitirá calcular los gastos en base a los actores involucrados en las tomas.

* La estructura para representar la solución es un array de *t* tomas, donde cada elemento representa el día que ser realizará la misma. Es una estructura que nos permitirá hacer diferentes asignaciones de los días sin complejidad, cambiar los días de una toma, intercambiarlos, etc. Calcular la limitación de máximo 6 tomas por días es una operación sencilla de acumular las ocurrencias de cada día.

Según el modelo para el espacio de soluciones<br>
(*)¿Cual es la función objetivo?

(*)¿Es un problema de maximización o minimización?

Respuesta: 

El objetivo es planificar las sesiones por día de manera que el gasto por los servicios de los actores de doblaje sea el menor posible. 


<u>Es un problema de minimización</u>

### importación de librerías

In [ ]:
import numpy as np
import pandas as pd

### Carga de datos del problema

In [15]:
# recuperar las tomas/actor desde csv
df = pd.read_csv('Datos problema doblaje (30 tomas, 10 actores).csv', delimiter=';')
df = df.drop('Toma', axis=1)
# generar la matriz toma x actor
problema = df.to_numpy()
print(problema)

[[1 1 1 1 1 0 0 0 0 0]
 [0 0 1 1 1 0 0 0 0 0]
 [0 1 0 0 1 0 1 0 0 0]
 [1 1 0 0 0 0 1 1 0 0]
 [0 1 0 1 0 0 0 1 0 0]
 [1 1 0 1 1 0 0 0 0 0]
 [1 1 0 1 1 0 0 0 0 0]
 [1 1 0 0 0 1 0 0 0 0]
 [1 1 0 1 0 0 0 0 0 0]
 [1 1 0 0 0 1 0 0 1 0]
 [1 1 1 0 1 0 0 1 0 0]
 [1 1 1 1 0 1 0 0 0 0]
 [1 0 0 1 1 0 0 0 0 0]
 [1 0 1 0 0 1 0 0 0 0]
 [1 1 0 0 0 0 1 0 0 0]
 [0 0 0 1 0 0 0 0 0 1]
 [1 0 1 0 0 0 0 0 0 0]
 [0 0 1 0 0 1 0 0 0 0]
 [1 0 1 0 0 0 0 0 0 0]
 [1 0 1 1 1 0 0 0 0 0]
 [0 0 0 0 0 1 0 1 0 0]
 [1 1 1 1 0 0 0 0 0 0]
 [1 0 1 0 0 0 0 0 0 0]
 [0 0 1 0 0 1 0 0 0 0]
 [1 1 0 1 0 0 0 0 0 1]
 [1 0 1 0 1 0 0 0 1 0]
 [0 0 0 1 1 0 0 0 0 0]
 [1 0 0 1 0 0 0 0 0 0]
 [1 0 0 0 1 1 0 0 0 0]
 [1 0 0 1 0 0 0 0 0 0]]


Diseña un algoritmo para resolver el problema por fuerza bruta

Respuesta

In [48]:
# costo para penalizar las soluciones que no cumplen la restricción
COSTO_RESTRICCION = 10000

# Calcula el costo de una solución.
def calcular_costo(problema, solucion):
    '''
    Calcula el costo de una solución.
    Si se exceden las 6 tomas en un mismo día, habrá un costo extra COSTO_RESTRICCION para penalizar la solución
    '''
    costo = 0
    
    # la cantidad de dias maxima es igual a la cantidad de tomas (es en el caso de que se haga una toma por dia)
    tomas = len(solucion)

    # agrupar las tomas por dia
    dias = [[] for i in range(tomas)]
    for t in range(tomas):
        d = solucion[t]
        dias[d].append(t)
        
    # calcular el costo por dia
    for d in range(len(dias)):
        # array donde cada elemento representa la cantidad de tomas en las que ese actor participará ese día.
        actores = [0] * problema.shape[1]
        # recorremos las tomas de ese día 
        for t in dias[d]:
            for a in range(len(actores)):
                actores[a] += problema[t, a]
        # calculamos el costo en base a la cantidad de actores
        for a in range(len(actores)):
            costo += 1 if actores[a] > 0 else 0 # incrementamos 1 el costo si el actor tiene almenos una toma
                
    # RESTRICCION: calcular un costo muy alto si se excede el maximo de 6 tomas
    for d in range(tomas):
        if len(dias[d]) > 6:
            costo += COSTO_RESTRICCION
    
    return costo

solucion = [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 29]
calcular_costo(problema, solucion)
    

10012

Calcula la complejidad del algoritmo por fuerza bruta

Respuesta

(*)Diseña un algoritmo que mejore la complejidad del algortimo por fuerza bruta. Argumenta porque crees que mejora el algoritmo por fuerza bruta

Respuesta

(*)Calcula la complejidad del algoritmo

Respuesta

Según el problema (y tenga sentido), diseña un juego de datos de entrada aleatorios

Respuesta

Aplica el algoritmo al juego de datos generado

Respuesta

Enumera las referencias que has utilizado(si ha sido necesario) para llevar a cabo el trabajo

Respuesta

Describe brevemente las lineas de como crees que es posible avanzar en el estudio del problema. Ten en cuenta incluso posibles variaciones del problema y/o variaciones al alza del tamaño

Respuesta